In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from textwrap import fill
import math

DATA_PATH = Path("../Processed/traffic_accidents_zh_clean.csv")
OUTPUT_PATH = Path("../../docs/pics/visualizations")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
def format_ch_number(value):
    return f"{float(value):,.0f}".replace(",", "’")


def ch_number_formatter(x, pos):
    return format_ch_number(x)


DARK_BG = "#0B1020"
TEXT_MAIN = "#CBD5E1"
TEXT_MUTED = "#94A3B8"
GRID = "#1E293B"
ACCENT_ORANGE = "#F28A63"

severity_colors = {
    "Accident with property damage": "#450b65",
    "Accident with light injuries": "#922760",
    "Accident with severe injuries": "#eb8709",
    "Accident with fatalities": "#edf19c"
}

In [ ]:
accidents_per_year = df["AccidentYear"].value_counts().sort_index()

y_min_data = accidents_per_year.min()
y_max_data = accidents_per_year.max()

y_min = math.floor((y_min_data - 300) / 500) * 500
y_max = math.ceil((y_max_data + 300) / 500) * 500

fig, ax = plt.subplots(figsize=(10.5, 5.8), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

ax.plot(
    accidents_per_year.index,
    accidents_per_year.values,
    linewidth=6,
    color=ACCENT_ORANGE,
    alpha=0.16,
    solid_capstyle="round",
    zorder=1
)

ax.plot(
    accidents_per_year.index,
    accidents_per_year.values,
    marker="o",
    markersize=7,
    linewidth=2.8,
    color=ACCENT_ORANGE,
    markerfacecolor=ACCENT_ORANGE,
    markeredgecolor="#FFD9C7",
    markeredgewidth=1.0,
    solid_capstyle="round",
    zorder=3
)

ax.fill_between(
    accidents_per_year.index,
    accidents_per_year.values,
    y_min,
    color=ACCENT_ORANGE,
    alpha=0.06,
    zorder=2
)

ax.set_title(
    "Traffic Accidents Over Time",
    fontsize=18,
    fontweight="bold",
    color="white",
    pad=28,
    loc="left"
)

ax.text(
    0.0,
    1.02,
    "Police-recorded traffic accidents in the Canton of Zurich, 2011–2025",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10.5,
    color=TEXT_MUTED
)

ax.set_xlabel("Year", color=TEXT_MAIN, labelpad=10, fontsize=11)
ax.set_ylabel("Number of accidents", color=TEXT_MAIN, labelpad=10, fontsize=11)

ax.set_ylim(y_min, y_max)
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_major_formatter(FuncFormatter(ch_number_formatter))

ax.set_xticks(accidents_per_year.index)
ax.set_xticklabels(accidents_per_year.index, color=TEXT_MAIN, fontsize=10)

ax.tick_params(axis="x", colors=TEXT_MAIN, labelsize=10, length=0)
ax.tick_params(axis="y", colors=TEXT_MUTED, labelsize=10, length=0)

ax.grid(axis="y", color=GRID, linestyle="--", linewidth=0.8, alpha=0.85)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.subplots_adjust(top=0.86)

plt.savefig(
    OUTPUT_PATH / "traffic_accidents_over_time.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

severity_counts = df["AccidentSeverityCategory_en"].value_counts()

plt.figure(figsize=(8, 4.5))
severity_counts.plot(kind="bar")

plt.title("Traffic accidents by severity category")
plt.xlabel("Severity category")
plt.ylabel("Number of accidents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(OUTPUT_PATH / "accidents_by_severity.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
vulnerable_counts = pd.DataFrame({
    "Participant group": ["Pedestrian", "Bicycle", "Motorcycle"],
    "Accidents": [
        int(df["AccidentInvolvingPedestrian"].sum()),
        int(df["AccidentInvolvingBicycle"].sum()),
        int(df["AccidentInvolvingMotorcycle"].sum())
    ]
})

vulnerable_colors = {
    "Pedestrian": "#EAD7A1",
    "Bicycle": "#6D28D9",
    "Motorcycle": "#DB2777"
}

fig, ax = plt.subplots(figsize=(7, 5), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

bars = ax.bar(
    vulnerable_counts["Participant group"],
    vulnerable_counts["Accidents"],
    color=[vulnerable_colors[group] for group in vulnerable_counts["Participant group"]],
    width=0.68,
    edgecolor="none"
)

ax.set_title(
    "Accidents Involving Vulnerable Road Users",
    fontsize=16,
    fontweight="bold",
    color="white",
    pad=24,
    loc="left"
)

ax.text(
    0.0,
    1.02,
    "Pedestrian, bicycle and motorcycle involvement in recorded accidents",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    color=TEXT_MUTED
)

ax.set_xlabel("")
ax.set_ylabel("Number of accidents", color=TEXT_MAIN, labelpad=10)

ax.tick_params(axis="x", colors=TEXT_MAIN, labelsize=10, length=0)
ax.tick_params(axis="y", colors=TEXT_MUTED, labelsize=9, length=0)

ax.yaxis.set_major_formatter(FuncFormatter(ch_number_formatter))
ax.yaxis.grid(True, color=GRID, linestyle="--", linewidth=0.8, alpha=0.8)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

max_val = vulnerable_counts["Accidents"].max()

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + max_val * 0.025,
        format_ch_number(height),
        ha="center",
        va="bottom",
        fontsize=10,
        color="#E2E8F0"
    )

ax.set_ylim(0, max_val * 1.18)

plt.subplots_adjust(top=0.84)

plt.savefig(
    OUTPUT_PATH / "accidents_involving_vulnerable_road_users.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

In [ ]:
# %%
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekday_hour = df.pivot_table(
    index="AccidentWeekDay_en",
    columns="AccidentHour",
    values="AccidentUID",
    aggfunc="count"
).fillna(0)

weekday_hour = weekday_hour.reindex(weekday_order)
weekday_hour = weekday_hour.reindex(columns=range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

im = ax.imshow(
    weekday_hour,
    aspect="auto",
    cmap="viridis"
)

ax.set_title(
    "Accidents by Weekday and Hour",
    fontsize=16,
    fontweight="bold",
    color="white",
    pad=28,
    loc="left"
)

ax.text(
    0.0,
    1.02,
    "Colour intensity shows recurring time windows with higher accident frequency",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    color=TEXT_MUTED
)

ax.set_xlabel("Hour of day", color=TEXT_MAIN, labelpad=12)
ax.set_ylabel("Weekday", color=TEXT_MAIN, labelpad=10)

ax.set_xticks(np.arange(24))
ax.set_xticklabels([str(h) for h in range(24)], color=TEXT_MAIN, fontsize=9)

ax.set_yticks(np.arange(len(weekday_hour.index)))
ax.set_yticklabels(weekday_hour.index, color=TEXT_MAIN, fontsize=10)

ax.tick_params(axis="both", length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
cbar.set_label("Number of accidents", color=TEXT_MAIN, labelpad=12)
cbar.ax.yaxis.set_tick_params(color=TEXT_MUTED)
plt.setp(cbar.ax.get_yticklabels(), color=TEXT_MUTED)
cbar.outline.set_visible(False)

plt.subplots_adjust(top=0.86)

plt.savefig(
    OUTPUT_PATH / "accidents_by_weekday_and_hour.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

In [ ]:
vulnerable_mapping = {
    "Pedestrian": "AccidentInvolvingPedestrian",
    "Bicycle": "AccidentInvolvingBicycle",
    "Motorcycle": "AccidentInvolvingMotorcycle"
}

vulnerable_type_frames = []

for group_name, column_name in vulnerable_mapping.items():
    temp = df[df[column_name] == True].copy()
    temp["Participant group"] = group_name
    vulnerable_type_frames.append(temp)

vulnerable_type_long = pd.concat(vulnerable_type_frames, ignore_index=True)

top_types = (
    vulnerable_type_long["AccidentType_en"]
    .value_counts()
    .head(8)
    .index
    .tolist()
)

vulnerable_type_long = vulnerable_type_long[
    vulnerable_type_long["AccidentType_en"].isin(top_types)
].copy()

vulnerable_type_counts = (
    vulnerable_type_long
    .groupby(["AccidentType_en", "Participant group"])
    .size()
    .reset_index(name="Accidents")
)

pivot_vulnerable_types = vulnerable_type_counts.pivot_table(
    index="AccidentType_en",
    columns="Participant group",
    values="Accidents",
    aggfunc="sum"
).fillna(0)

pivot_vulnerable_types = pivot_vulnerable_types.loc[top_types]
pivot_vulnerable_types = pivot_vulnerable_types[["Pedestrian", "Bicycle", "Motorcycle"]]

wrapped_labels = [fill(label, width=30) for label in pivot_vulnerable_types.index]

fig, ax = plt.subplots(figsize=(11, 6.8), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

left = np.zeros(len(pivot_vulnerable_types))

for group in ["Pedestrian", "Bicycle", "Motorcycle"]:
    values = pivot_vulnerable_types[group].values
    ax.barh(
        wrapped_labels,
        values,
        left=left,
        color=vulnerable_colors[group],
        label=group,
        height=0.72,
        edgecolor=DARK_BG,
        linewidth=0.5
    )
    left += values

ax.invert_yaxis()

ax.set_title(
    "Accident Types by Vulnerable Road User",
    fontsize=16,
    fontweight="bold",
    color="white",
    pad=26,
    loc="left"
)

ax.text(
    0.0,
    1.02,
    "Most frequent accident types split by involved road user group",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    color=TEXT_MUTED
)

ax.set_xlabel("Number of accidents", color=TEXT_MAIN, labelpad=12)
ax.set_ylabel("")

ax.tick_params(axis="x", colors=TEXT_MUTED, labelsize=9, length=0)
ax.tick_params(axis="y", colors=TEXT_MAIN, labelsize=9, length=0, pad=8)

ax.xaxis.set_major_formatter(FuncFormatter(ch_number_formatter))
ax.xaxis.grid(True, color=GRID, linestyle="--", linewidth=0.8, alpha=0.8)
ax.yaxis.grid(False)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

legend = ax.legend(
    title="Road user group",
    frameon=False,
    labelcolor=TEXT_MAIN,
    title_fontsize=10,
    fontsize=9,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.25),
    ncol=3
)

legend.get_title().set_color(TEXT_MAIN)

plt.subplots_adjust(top=0.86, bottom=0.25, left=0.34)

plt.savefig(
    OUTPUT_PATH / "accident_types_by_vulnerable_road_user.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

In [ ]:
road_severity = pd.crosstab(
    df["RoadType_en"],
    df["AccidentSeverityCategory_en"]
)

road_severity = road_severity.loc[
    road_severity.sum(axis=1).sort_values(ascending=False).index
]

severity_order = road_severity.sum(axis=0).sort_values(ascending=False).index.tolist()
road_severity = road_severity[severity_order]

available_colors = [
    severity_colors.get(col, "#94A3B8")
    for col in road_severity.columns
]

fig, ax = plt.subplots(figsize=(11, 6.2), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

road_severity.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=available_colors,
    width=0.72,
    edgecolor=DARK_BG,
    linewidth=0.6
)

ax.set_title(
    "Accident Severity by Road Type",
    fontsize=17,
    fontweight="bold",
    color="white",
    pad=22,
    loc="left"
)

ax.text(
    0.0,
    1.02,
    "Total accident volume and severity mix across road types",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    color=TEXT_MUTED
)

ax.set_xlabel("")
ax.set_ylabel("Number of accidents", color=TEXT_MAIN, labelpad=10)

wrapped_road_labels = [fill(label.get_text(), width=12) for label in ax.get_xticklabels()]
ax.set_xticklabels(wrapped_road_labels)

ax.tick_params(
    axis="x",
    colors=TEXT_MAIN,
    labelsize=8,
    rotation=0,
    length=0
)

for label in ax.get_xticklabels():
    label.set_ha("center")

ax.tick_params(
    axis="y",
    colors=TEXT_MUTED,
    labelsize=9,
    length=0
)

ax.yaxis.set_major_formatter(FuncFormatter(ch_number_formatter))
ax.yaxis.grid(True, color="white", linewidth=0.8, alpha=0.28)
ax.xaxis.grid(False)
ax.set_axisbelow(False)

for spine in ax.spines.values():
    spine.set_visible(False)

legend = ax.legend(
    title="Severity category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    labelcolor=TEXT_MAIN,
    title_fontsize=10,
    fontsize=9
)

legend.get_title().set_color(TEXT_MAIN)

plt.subplots_adjust(top=0.84, right=0.80)

plt.savefig(
    OUTPUT_PATH / "accident_severity_by_road_type.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(8, 8), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

hb = ax.hexbin(
    df["AccidentLocation_CHLV95_E"],
    df["AccidentLocation_CHLV95_N"],
    gridsize=120,
    mincnt=1,
    cmap="inferno",
    bins="log",
    linewidths=0,
    alpha=0.95
)

ax.set_title(
    "Traffic Accident Hotspots",
    fontsize=18,
    fontweight="bold",
    color="white",
    pad=20,
    loc="left"
)

ax.text(
    0.0,
    1.01,
    "Spatial concentration of recorded accidents using CHLV95 coordinates",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10.5,
    color=TEXT_MUTED
)

ax.set_aspect("equal", adjustable="box")

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(hb, ax=ax, shrink=0.72, pad=0.02)
cbar.set_label("Accident density", color=TEXT_MAIN)
cbar.ax.yaxis.set_tick_params(color=TEXT_MUTED)
plt.setp(cbar.ax.get_yticklabels(), color=TEXT_MUTED)

cbar.ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: format_ch_number(x) if x >= 1 else "")
)

cbar.outline.set_visible(False)

plt.savefig(
    OUTPUT_PATH / "traffic_accident_hotspots.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()